# 07 — Fold 3 Final Production Simulation

**This notebook is run exactly once. It is never rerun. The moment Fold 3 results are
visible, the pipeline is locked regardless of what they show.**

Fold 3 is not a model tuning step. It is a production simulation: does the full
pipeline — routing, forecasting, uncertainty quantification, and inventory
simulation — behave consistently on unseen data?

**Inputs (Fold 2 lock-ins, reused verbatim, not re-derived):**
- `winner_decision_fold2.pkl` — 06b, production model identity
- `regime_thresholds.pkl` — 06c, ADI/CV² thresholds (1.32, 0.49)
- `conformal_residuals_fold2.pkl` — 06e, locked per-regime service level / alpha
- `features_train_v2.parquet` (train, ends 2015-01-31) and `features_val_v2.parquet`
  (val, 2015-02-01 -> 2016-01-31) — 04b, two separate files, not one date-split
- `feature_cols_v2.pkl` — 04b
- 06g Sections 7 and 9's cost formulas and parameters (safety stock, review period,
  lead time, `52/N_WEEKS_VAL` annualization) — reused directly in Section 5b, not
  re-derived

**Outputs:**
- `tweedie_optimized_fold3.txt`, `sku_regimes_fold3.parquet`,
  `conformal_residuals_fold3.pkl`, `final_predictions_fold3.parquet`,
  `simulation_results_fold3.parquet`
- `dynamic_cost_sensitivity_fold3.parquet` (Section 5b, mirrors 06g Section 8)
- `static_vs_dynamic_headtohead_fold3.parquet` (Section 5b, mirrors 06g Section 9 —
  this is the app's headline source)

---

## Section 1 — Pre-Flight Checklist

Confirm every locked decision below BEFORE any Fold 3 data is touched. Nothing in
this section re-derives a decision — it only checks that what's on disk still
matches what prior notebooks locked, and defines the Fold 3 train/val split.


In [3]:
# -- 07 Section 1: Pre-Flight Checklist --------------------------------------------------
import numpy as np
import pandas as pd
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR    = '../data/processed'
CALIBRATION_DIR  = f'{PROCESSED_DIR}/calibration'
SEGMENTATION_DIR = f'{PROCESSED_DIR}/segmentation'
MODELS_DIR       = f'{PROCESSED_DIR}/models'
PREDICTIONS_DIR  = f'{PROCESSED_DIR}/predictions'
FEATURES_DIR     = f'{PROCESSED_DIR}/features'

FOLD3_VAL_START = pd.Timestamp('2015-02-01')

# -- Locked decisions from pre-Fold 3 work -- confirm, do not re-derive ------------------
# 06d tested a direct 7-day target and asymmetric loss; both failed the adoption gate.
# Neither was adopted -- PRODUCTION_MODEL is the unmodified 06b baseline, saved under 06d's
# output filename (tweedie_optimized_fold2.txt) but architecturally identical to 06b.
PRODUCTION_MODEL  = 'tweedie_baseline'
HYSTERESIS_WEEKS  = 4

# -- Confirm 06b's winner decision is what this notebook assumes ------------------------
winner_path = f'{CALIBRATION_DIR}/winner_decision_fold2.pkl'
assert os.path.exists(winner_path), f'Missing {winner_path} -- 06b winner lock-in not found.'
with open(winner_path, 'rb') as f:
    winner = pickle.load(f)

print('-- 06b winner decision (locked) --')
print(f"  Model:   {winner.get('WINNER_MODEL')}")
print(f"  Variant: {winner.get('WINNER_VARIANT')}")
assert winner.get('WINNER_MODEL') == 'tweedie', (
    f"winner_decision_fold2.pkl says {winner.get('WINNER_MODEL')!r}, not 'tweedie' -- "
    f"PRODUCTION_MODEL assumption above is stale, stop and check 06b before proceeding."
)
# NOTE: winner['WINNER_VARIANT'] is 'raw', not 'suppressed' -- new_plan.md's Immediate-
# Next-Step draft said 'suppressed'; the pkl is the real locked artifact and takes
# precedence. Plan doc update pending confirmation against 06b's own notebook.

# -- Regime thresholds -- locked in 06c, reused verbatim for Fold 3 reclassification ----
thresholds_path = f'{SEGMENTATION_DIR}/regime_thresholds.pkl'
assert os.path.exists(thresholds_path), f'Missing {thresholds_path} -- 06c output not found.'
with open(thresholds_path, 'rb') as f:
    thresholds_meta = pickle.load(f)

# BUG FIXED (v1 of this cell): compared the whole metadata dict (keys adi_threshold/
# cv2_threshold plus counts/fold/train_end) against a plain {'adi', 'cv2'} dict -- always
# false regardless of the real values. Extract the two threshold values explicitly instead.
REGIME_THRESHOLDS = {
    'adi': thresholds_meta['adi_threshold'],
    'cv2': thresholds_meta['cv2_threshold'],
}
assert REGIME_THRESHOLDS == {'adi': 1.32, 'cv2': 0.49}, (
    f'REGIME_THRESHOLDS on disk ({REGIME_THRESHOLDS}) does not match new_plan.md -- '
    f'Section 2 reclassification would use different thresholds than every prior fold.'
)
print(f'Regime thresholds (locked, 06c): {REGIME_THRESHOLDS}')

FOLD2_REGIME_COUNTS = thresholds_meta['regime_counts']
print(f'Fold 2 regime counts (for Fold 2 vs Fold 3 comparison in Section 2): {FOLD2_REGIME_COUNTS}')

# -- Conformal config -- locked in 06e, refit ON Fold 3 data in Section 4, same method/alpha
conformal_path = f'{CALIBRATION_DIR}/conformal_residuals_fold2.pkl'
assert os.path.exists(conformal_path), f'Missing {conformal_path} -- 06e output not found.'
with open(conformal_path, 'rb') as f:
    conformal_fold2 = pickle.load(f)

SERVICE_LEVEL_BY_REGIME = conformal_fold2['default_service_level']
CONFORMAL_ALPHA = {
    regime: round(1 - int(level[1:]) / 100, 2)
    for regime, level in SERVICE_LEVEL_BY_REGIME.items()
}
print(f'Locked service levels by regime (06e): {SERVICE_LEVEL_BY_REGIME}')
print(f'Corresponding conformal alpha by regime: {CONFORMAL_ALPHA}')
print(f'Hysteresis: {HYSTERESIS_WEEKS} weeks (locked, 06c)')

# -- Load Fold 3 train/val -- TWO SEPARATE FILES, not a date-split of one ----------------
# features_train_v2.parquet ends 2015-01-31 and never contains Fold 3's val window.
# features_val_v2.parquet holds 2015-02-01 -> 2016-01-31 instead (confirmed by direct
# inspection: same schema, all 30,490 SKUs, 365 unique dates) -- Fold 3 train/val are
# sourced from two different files by design, not filtered from one shared file.
fold3_train = pd.read_parquet(f'{FEATURES_DIR}/features_train_v2.parquet')
fold3_train['date'] = pd.to_datetime(fold3_train['date'])

fold3_val = pd.read_parquet(f'{FEATURES_DIR}/features_val_v2.parquet')
fold3_val['date'] = pd.to_datetime(fold3_val['date'])

assert set(fold3_val.columns) == set(fold3_train.columns), (
    'features_val_v2.parquet schema does not match features_train_v2.parquet -- cannot '
    'safely apply the same feature_cols/model to both.'
)

print(f"\nFold 3 train: {fold3_train['date'].min().date()} -> {fold3_train['date'].max().date()} "
      f"({fold3_train['date'].nunique():,} days, {len(fold3_train):,} rows)")
print(f"Fold 3 val:   {fold3_val['date'].min().date()} -> {fold3_val['date'].max().date()} "
      f"({fold3_val['date'].nunique():,} days, {len(fold3_val):,} rows)")

# -- Extra safety check beyond the plan's two asserts: since train/val now come from two
# independently-loaded files rather than one date-split, confirm no date appears in both --
# a stale or duplicated file could otherwise leak val dates into train silently. ---------
overlap_dates = set(fold3_train['date'].unique()) & set(fold3_val['date'].unique())
assert not overlap_dates, f'{len(overlap_dates)} date(s) appear in both train and val -- leakage.'

# -- Pre-flight assertions (verbatim from new_plan.md) -----------------------------------
assert fold3_val['date'].min() == pd.Timestamp('2015-02-01'), 'Fold 3 start mismatch'
assert fold3_train['date'].max() < pd.Timestamp('2015-02-01'), 'Leakage detected'
print('\nPre-flight assertions passed.')


-- 06b winner decision (locked) --
  Model:   tweedie
  Variant: raw
Regime thresholds (locked, 06c): {'adi': 1.32, 'cv2': 0.49}
Fold 2 regime counts (for Fold 2 vs Fold 3 comparison in Section 2): {'intermittent': 14268, 'smooth': 8389, 'lumpy': 7003, 'erratic': 830}
Locked service levels by regime (06e): {'smooth': 'q80', 'erratic': 'q80'}
Corresponding conformal alpha by regime: {'smooth': 0.2, 'erratic': 0.2}
Hysteresis: 4 weeks (locked, 06c)

Fold 3 train: 2011-02-02 -> 2015-01-31 (1,460 days, 28,699,814 rows)
Fold 3 val:   2015-02-01 -> 2016-01-31 (365 days, 11,128,850 rows)

Pre-flight assertions passed.


#### Section 1 Findings — Pre-Flight Checklist

**Status: LOCKED**, with one open item flagged below (does not block Section 2).

**Confirmed against actual saved artifacts, not assumed:**
- `winner_decision_fold2.pkl`: `WINNER_MODEL='tweedie'`, `WINNER_VARIANT='raw'`.
- `regime_thresholds.pkl`: `adi=1.32`, `cv2=0.49` — matches every prior fold. Fold 2 regime
  counts captured (`intermittent`: 14,268, `smooth`: 8,389, `lumpy`: 7,003, `erratic`: 830)
  for the Fold 2 vs. Fold 3 stability comparison in Section 2.
- `conformal_residuals_fold2.pkl`: locked service levels `{'smooth': 'q80', 'erratic': 'q80'}`,
  i.e. `alpha=0.20` for both. Lumpy/Intermittent have no conformal wrapper by design (native
  Croston/TSB uncertainty, per Decision 2) — `CONFORMAL_ALPHA` intentionally only covers
  Smooth/Erratic.
- Fold 3 train/val: sourced from two separate files, not a date-split of one —
  `features_train_v2.parquet` (2011-02-02 → 2015-01-31, 28,699,814 rows) and
  `features_val_v2.parquet` (2015-02-01 → 2016-01-31, 11,128,850 rows). Schema match and
  zero date-overlap both confirmed. Both plan-specified pre-flight assertions passed.

**Two bugs found and fixed in this cell before it produced a trustworthy result:**
1. `REGIME_THRESHOLDS` comparison originally checked the *entire* `regime_thresholds.pkl`
   metadata dict (which also carries `regime_counts`, `routing_counts`, `train_end`, etc.)
   against a plain `{'adi': 1.32, 'cv2': 0.49}` — always false regardless of the real
   threshold values. Fixed by extracting `adi_threshold`/`cv2_threshold` explicitly.
2. `fold3_val` was originally built by date-filtering `features_train_v2.parquet`, which
   silently produced an empty frame — that file ends 2015-01-31 and was never meant to
   contain Fold 3's val window. Confirmed by direct inspection that `features_val_v2.parquet`
   holds 2015-02-01 → 2016-01-31 instead, same schema, all 30,490 SKUs. Fixed by loading
   train and val from their two separate source files.

**Open item, not blocking:** `WINNER_VARIANT='raw'` on disk conflicts with `new_plan.md`'s
Immediate-Next-Step draft, which specifies `'suppressed'`. The pkl is the real locked
artifact and takes precedence for this notebook. `new_plan.md`'s text has not yet been
updated to match — pending direct confirmation against `06b_model_selection.ipynb`'s own
findings cell before that edit is applied.

No other open items. Proceeding to Section 2.
